# SONA Overnight Trust-Region Sweep

This notebook is meant to run unattended for several hours on Kaggle T4. It trains the `attention` backbone once, then evaluates a large trust-region online-adaptation grid on 300 VoiceBank-DEMAND test clips.

Goal: find a setting that improves PESQ while keeping SI-SDRi/STOI nearly unchanged.

In [ ]:
%pip install -q pesq pystoi ptflops

import torch
import torchaudio

print('torch', torch.__version__)
print('torchaudio', torchaudio.__version__)
print('cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Turn on GPU accelerator before running.'

In [ ]:
import glob
from pathlib import Path

def find_repo():
    hits = glob.glob('/kaggle/input/**/scripts/kaggle_tune_online.py', recursive=True)
    hits = sorted(hits, key=lambda p: ('sona' not in p.lower(), len(p)))
    return str(Path(hits[0]).parents[1]) if hits else None

def has_vbd_layout(root):
    root = Path(root)
    needed = ['clean_trainset_wav', 'noisy_trainset_wav', 'clean_testset_wav', 'noisy_testset_wav']
    return all((root / name).is_dir() for name in needed)

def find_vbd():
    hits = glob.glob('/kaggle/input/**/noisy_trainset_wav', recursive=True)
    roots = [str(Path(h).parent) for h in hits if has_vbd_layout(Path(h).parent)]
    return sorted(roots, key=len)[0] if roots else None

REPO_DIR = find_repo()
VBD_ROOT = find_vbd()
OUT_DIR = '/kaggle/working/results_overnight_trust_region'

# If needed, set manually from Kaggle's right panel.
# REPO_DIR = '/kaggle/input/sona-repo'
# VBD_ROOT = '/kaggle/input/voicebank-demand-16k'

assert REPO_DIR, 'Updated sona-repo not found. Upload refreshed sona.zip first.'
assert VBD_ROOT, 'VoiceBank-DEMAND folders not found.'

txt = (Path(REPO_DIR) / 'scripts' / 'kaggle_tune_online.py').read_text(encoding='utf-8')
assert '--max-gain-deltas' in txt, 'sona-repo is old. Upload the refreshed sona.zip with trust-region controls.'

print('REPO_DIR =', REPO_DIR)
print('VBD_ROOT =', VBD_ROOT)
print('OUT_DIR  =', OUT_DIR)

In [ ]:
# Overnight sweep: one attention training run + many online settings.
# Expected time: roughly 4-6 hours on Kaggle T4, depending on runtime speed.
LIMIT_TRAIN = 2000
LIMIT_TEST = 300
EPOCHS = 40
SUP_MODELS = 'attention'
ADAPTERS = 'scalar,lowrank'
ANCHORS = '0.5,1.0,2.0,4.0'
STEPS = '1,2,3,4,6'
MAX_GAIN_DELTAS = '0.01,0.02,0.05,0.1,none'
SMOOTH_TIME_WEIGHT = 0.0
SMOOTH_BAND_WEIGHT = 0.0
LR = 0.1

print(dict(
    LIMIT_TRAIN=LIMIT_TRAIN,
    LIMIT_TEST=LIMIT_TEST,
    EPOCHS=EPOCHS,
    SUP_MODELS=SUP_MODELS,
    ADAPTERS=ADAPTERS,
    ANCHORS=ANCHORS,
    STEPS=STEPS,
    MAX_GAIN_DELTAS=MAX_GAIN_DELTAS,
    SMOOTH_TIME_WEIGHT=SMOOTH_TIME_WEIGHT,
    SMOOTH_BAND_WEIGHT=SMOOTH_BAND_WEIGHT,
    LR=LR,
))

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, f'{REPO_DIR}/scripts/kaggle_tune_online.py',
    '--repo', REPO_DIR,
    '--vbd-root', VBD_ROOT,
    '--out', OUT_DIR,
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-test', str(LIMIT_TEST),
    '--epochs', str(EPOCHS),
    '--sup-models', SUP_MODELS,
    '--adapters', ADAPTERS,
    '--anchors', ANCHORS,
    '--steps', STEPS,
    '--max-gain-deltas', MAX_GAIN_DELTAS,
    '--smooth-time-weight', str(SMOOTH_TIME_WEIGHT),
    '--smooth-band-weight', str(SMOOTH_BAND_WEIGHT),
    '--lr', str(LR),
]

print(' '.join(cmd), '\n')
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
assert proc.returncode == 0, f'kaggle_tune_online.py exited {proc.returncode}'

In [ ]:
# Rank settings by useful tradeoff, not PESQ alone.
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path(OUT_DIR) / 'summary.csv')
base = df[df.method == 'neural_attention'].iloc[0]
online = df[df.method.str.startswith('attention_online_')].copy()

for col in ['si_sdr_i', 'pesq', 'stoi', 'estoi', 'lsd', 'rtf']:
    online['delta_' + col] = online[col] - base[col]

# Soft score: reward PESQ, penalize SI-SDRi drop beyond 0.15 dB, STOI drop beyond 0.001, and LSD worsening.
online['safe_score'] = (
    online['delta_pesq']
    - 0.05 * (-(online['delta_si_sdr_i'] + 0.15)).clip(lower=0)
    - 20.0 * (-(online['delta_stoi'] + 0.001)).clip(lower=0)
    - 0.01 * (online['delta_lsd']).clip(lower=0)
)

cols = [
    'method', 'si_sdr_i', 'pesq', 'stoi', 'estoi', 'lsd', 'rtf',
    'delta_si_sdr_i', 'delta_pesq', 'delta_stoi', 'delta_estoi', 'delta_lsd', 'safe_score'
]
ranked = online.sort_values(['safe_score', 'delta_pesq'], ascending=[False, False])[cols]
display(ranked.head(25))

print('\nFrozen baseline:')
display(df[df.method == 'neural_attention'][['method', 'si_sdr_i', 'pesq', 'stoi', 'estoi', 'lsd', 'rtf']])

print('\nBest by raw PESQ:')
display(online.sort_values('pesq', ascending=False)[cols].head(10))

print('\nBest with SI-SDRi drop <= 0.2 dB and STOI drop <= 0.0015:')
safe = online[(online.delta_si_sdr_i >= -0.2) & (online.delta_stoi >= -0.0015)]
display(safe.sort_values(['delta_pesq', 'delta_si_sdr_i'], ascending=[False, False])[cols].head(20))

ranked.to_csv('/kaggle/working/overnight_ranked_configs.csv', index=False)

In [ ]:
# Bundle artifacts for download.
import glob
import os
import shutil
from IPython.display import FileLink, display

ART = '/kaggle/working/sona_overnight_trust_region_artifacts'
shutil.copy('/kaggle/working/overnight_ranked_configs.csv', f'{OUT_DIR}/overnight_ranked_configs.csv')
shutil.make_archive(ART, 'zip', OUT_DIR)
print(f'Wrote {ART}.zip ({os.path.getsize(ART + ".zip") / 1e6:.1f} MB)')
for f in sorted(glob.glob(OUT_DIR + '/**', recursive=True)):
    if os.path.isfile(f):
        print('  ', os.path.relpath(f, OUT_DIR), f'({os.path.getsize(f) / 1e3:.0f} KB)')
display(FileLink(ART + '.zip'))